# Datasets - EDA

## Environment

### Working directory

In [ ]:
import os
os.chdir(os.getcwd() + "/../..")  # According to .ipynb directory

### Imports

In [ ]:
from scipy.spatial.distance import cdist
from typing import Dict
from typing_extensions import Self

import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

from utils.plots.pitch import drawPitch

### Settings

In [ ]:
pio.renderers.default = "notebook"

## Data

### Path

In [ ]:
file_path = "data/data/cleaned/{folder}/10510.parquet"

### Read dataframes

#### 1. Game

In [ ]:
game_df = pd.read_parquet(file_path.format(folder="games"))
game_df.head()

#### 2. Lineup

In [ ]:
lineup_df = pd.read_parquet(file_path.format(folder="lineups"))
lineup_df.head()

#### 3. Tracking

In [ ]:
tracking_df = pd.read_parquet(file_path.format(folder="tracking"))
tracking_df.head()

## Analysis

### Plot players in pitch from a specific frame

In [ ]:
class Tracking:
    """..."""

    def __init__(self):
        """..."""
        self.data: pd.DataFrame
        self.game_info: Dict
        self.pitch_length: int
        self.pitch_width: int
        self.pitch_layout: go.Layout
    
    def pre_processing(self, game_df: pd.DataFrame, lineup_df: pd.DataFrame, tracking_df: pd.DataFrame) -> Self:
        """..."""
        merged_df = lineup_df.merge(game_df, on="game.id")
        merged_df["teamGame"] = np.where(
            merged_df["team.id"] == merged_df["homeTeam.id"], "homePlayers", "awayPlayers"
        )
        merged_df = merged_df[
            ["teamGame", "jerseyNum", "player.nickname", "positionGroupType", "awayTeam.shortName", "homeTeam.shortName"]
        ]
        self.data = tracking_df.merge(merged_df, on=["jerseyNum", "teamGame"], how="left")
        self.game_info = game_df.iloc[0].to_dict()
        self.pitch_length = int(self.game_info["stadium.pitches.length"])
        self.pitch_width = int(self.game_info["stadium.pitches.width"])
        self.data["x.raw"] = self.data["x.raw"] + (self.pitch_length / 2)
        self.data["y.raw"] = self.data["y.raw"] + (self.pitch_width / 2)
        conditions = [
            self.data["teamGame"] == "awayPlayers",
            self.data["teamGame"] == "homePlayers",
            self.data["teamGame"] == "balls"
        ]
        self.data["primaryColor"] = np.select(
            conditions,
            [
                self.game_info["awayTeamKit.primaryColor"],
                self.game_info["homeTeamKit.primaryColor"],
                "#FAF9F6"
            ],
            default=None
        )
        self.data["team"] = np.select(
            conditions,
            [
                self.game_info[f"awayTeam.shortName"],
                self.game_info[f"homeTeam.shortName"],
                None
            ],
            default=None
        )
        self.data["markerSize"] = np.where(self.data["teamGame"] == "balls", 6, 12)
        return self
    
    def create_pitch_layout(self) -> Self:
        """..."""
        self.pitch_layout = go.Layout(
            margin={"l": 0, "r": 0, "b": 0, "t": 0},
            xaxis={"range": [0, self.pitch_length], "autorange": False, "showticklabels": False, "showgrid": False},
            yaxis={"range": [0, self.pitch_width], "autorange": False, "showticklabels": False, "showgrid": False},
            plot_bgcolor="#86dd48",
            shapes=drawPitch(x=self.pitch_length, y=self.pitch_width, num_zones_x=6, num_zones_y=4)
        )
        return self

    def plot_frame(self, frame: int) -> None:
        """..."""
        frame_df = self.data[self.data["frameNum"] == frame].copy()
        frame_df["ballDistance"] = np.round(
            cdist(
                frame_df[["x.raw", "y.raw"]].values,
                [frame_df[frame_df["teamGame"] == "balls"][["x.raw", "y.raw"]].values[0]]
            ).flatten(),
            1
        )
        frame_df["label"] = np.where(
            frame_df["teamGame"] == "balls",
            "Ball",
            "Player: " + frame_df["player.nickname"]
            + "<br>Team: " + frame_df["team"]
            + "<br>Position: " + frame_df["positionGroupType"]
            + "<br>Visibility: " + frame_df["visibility.raw"]
            + "<br>Confidence: " + frame_df["confidence.raw"]
            + "<br>Ball Distance: " + frame_df["ballDistance"].astype(str)
        )
        fig = go.Figure()
        fig.add_trace(
            go.Scatter(
                x=frame_df["x.raw"],
                y=frame_df["y.raw"],
                mode="markers",
                text=frame_df["label"],
                textposition="top center",
                marker=dict(
                    size=frame_df["markerSize"],
                    color=frame_df["primaryColor"],
                    line=dict(
                        color="#000000",
                        width=2
                    )
                ),
                hoverinfo="text"
            )
        )
        fig.update_layout(self.pitch_layout)
        fig.show()

In [ ]:
tracking = Tracking().pre_processing(game_df, lineup_df, tracking_df).create_pitch_layout()

In [ ]:
tracking.plot_frame(12000)